In [1]:
# Walmart engineering teams have many AI use cases, but the important question is:
#     “Should we solve this problem using traditional software, a fixed workflow, a hybrid architecture, or a full AI agent?”

# The notebook is trying to prevent a common mistake: using an AI agent just because an agent is technically capable of solving the problem.

# For example, imagine Walmart wants to automate a product return. The system mainly needs to check the purchase date, product category, return policy, and eligibility. These are predictable rules. Using a full AI agent here could unnecessarily increase cost, latency, complexity, and risk.

# But consider supplier risk intelligence. Walmart may need to read news, financial reports, regulatory information, and internal supplier data and then reason about different risks. Here, the steps cannot always be predefined, so an agent may make more sense.

# The notebook therefore creates a simple architecture decision framework based on five questions:
# 1. Task complexity – Is the problem simple and predictable, or does it require dynamic reasoning?
# 2. Latency tolerance – Does Walmart need an answer immediately, or can the system take more time?
# 3. Cost ceiling – Is this running millions of times where every cent matters, or only occasionally?
# 4. Risk tolerance – What happens if the AI gives an incorrect answer?
# 5. Update frequency – Does the business logic stay stable, or does it change frequently?

# Based on these scores, the system recommends one of four architectures: Traditional Software → Workflow → Hybrid Agent + Workflow → Agent.

In [2]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print('Environment loaded.')
print(f'OpenAI client ready: {client is not None}')

Environment loaded.
OpenAI client ready: True


## The Architecture Decision Problem

Every AI project begins with a question the team usually answers by gut feel: *should this be an agent?*
In production, gut feel costs money. A mis-classified problem -- one that gets an agent when a workflow
was sufficient -- adds latency, cost, observability complexity, and failure surface area for no benefit.

There are four architecture archetypes:

| Archetype | When to use | When NOT to use |
|-----------|-------------|------------------|
| **Traditional software** | Deterministic rules, structured input/output, no ambiguity | Tasks requiring language understanding or dynamic reasoning |
| **Workflow (chain)** | Known steps, fixed sequence, no branching on content | Tasks requiring mid-process decisions based on intermediate results |
| **Hybrid (Agent + Workflow)** | High reasoning need on some steps, hard latency or cost constraint on others | Tasks that are uniformly simple or uniformly complex |
| **Agent** | Open-ended reasoning, dynamic tool selection, multi-step planning under uncertainty | Latency-critical tasks, fully deterministic tasks, tasks where all steps are known upfront |

The decision is not about capability -- an agent can technically do anything a workflow can.
The decision is about cost, latency, predictability, and operational risk.

The hybrid archetype exists because real production systems frequently face a contradiction:
the task is complex enough to need intelligent reasoning, but the volume or latency budget prevents
running a full agent on every query. The correct response to that contradiction is not to compromise
by picking either pure option -- it is to design a system where a workflow backbone handles
predictable steps and an agent sub-component is invoked only where dynamic reasoning is required.

## The Five-Axis Decision Model

Score each axis 1-5 for your use case. Higher scores push toward agents.

| Axis | Score 1 | Score 5 |
|------|---------|----------|
| **Task Complexity** | Single step, fully deterministic | Multi-step, dynamic branching required |
| **Latency Requirement** | Real-time (< 500ms) | Batch or async acceptable (> 10s) |
| **Cost Ceiling** | Pennies per query | Dollars per query acceptable |
| **Risk Tolerance** | Zero hallucination tolerance | Some imprecision acceptable if caught downstream |
| **Update Frequency** | Logic changes rarely | Requirements change weekly |

In [ ]:
# 1. Task Complexity
# Score 1: Very simple, fixed steps.
# Score 5: Many steps, and the next step depends on what happens.

# 2. Latency Requirement
# Score 1: Must respond almost immediately, for example under 500 ms.
# Score 5: Taking 10 seconds or more is acceptable.

# 3. Cost Ceiling
# Score 1: Cost must be extremely low.
# Score 5: Higher cost is acceptable if the task is valuable.

# 4. Risk Tolerance
# Score 1: Almost no mistakes are acceptable.
# Score 5: Some error is acceptable because another system or human will check it.

# 5. Update Frequency
# Score 1: Rules rarely change.
# Score 5: Requirements change very frequently.

In [3]:
AXES = ['task_complexity', 'latency_tolerance', 'cost_ceiling', 'risk_tolerance', 'update_frequency']

AXIS_DESCRIPTIONS = {
    'task_complexity':    'How many distinct decision points exist in the task?',
    'latency_tolerance':  'How much time can the system take to respond?',
    'cost_ceiling':       'What is the acceptable cost per query?',
    'risk_tolerance':     'How much imprecision is acceptable in the output?',
    'update_frequency':   'How often do requirements or logic change?',
}

THRESHOLDS = {
    'traditional': (5, 12),   # total score range for pure archetypes
    'workflow':    (13, 18),
    'agent':       (19, 25),
}

# Hybrid override constants.
# A task triggers hybrid when it demands high reasoning (task_complexity >= 4)
# but faces a hard operational constraint on latency or cost (score <= 2).
# This combination means neither a pure agent nor a pure workflow is sufficient.
HYBRID_COMPLEXITY_MIN = 4 # The task must have a complexity score of at least 4 out of 5 before the notebook considers Hybrid architecture.
HYBRID_CONSTRAINT_MAX = 2 # If latency tolerance or cost ceiling is 2 or lower, the system has a strong operational constraint.


def recommend_architecture(scores: dict) -> dict:
    total = sum(scores.values())
    complexity = scores.get('task_complexity', 0)
    latency = scores.get('latency_tolerance', 0)
    cost = scores.get('cost_ceiling', 0)

    # Hybrid override is evaluated before the score bands.
    # When a task is both complex (needing intelligent reasoning on some steps)
    # and constrained (tight latency or cost preventing full agent deployment),
    # the score-band result would be misleading -- the correct architecture is
    # a workflow backbone with an agent sub-component on the complex steps only.
    if complexity >= HYBRID_COMPLEXITY_MIN and (
        latency <= HYBRID_CONSTRAINT_MAX or cost <= HYBRID_CONSTRAINT_MAX
    ):
        return {
            'architecture': 'Hybrid (Agent + Workflow)',
            'total_score': total,
            'hybrid_override': True,
            'rationale': (
                'High task complexity coexists with a strict latency or cost constraint. '
                'Neither a pure agent nor a pure workflow fits: agent overhead is unacceptable '
                'on every step, but deterministic logic cannot handle all branches. '
                'A hybrid architecture uses a workflow backbone to route and handle predictable '
                'steps, with an agent sub-component invoked only for steps requiring dynamic reasoning.'
            ),
        }

    if total <= 12:
        arch = 'Traditional Software'
        rationale = (
            'Low complexity, strict latency, or low cost ceiling. '
            'A rule-based or deterministic pipeline is faster, cheaper, and more predictable.'
        )
    elif total <= 18:
        arch = 'Workflow (Chain)'
        rationale = (
            'Moderate complexity with known steps. '
            'A fixed-sequence workflow gives you LLM capability without agent overhead.'
        )
    else:
        arch = 'Agent'
        rationale = (
            'High complexity, dynamic branching, or rapidly changing requirements. '
            'An agent is justified -- but implement observability and cost controls from day one.'
        )
    return {'architecture': arch, 'total_score': total, 'hybrid_override': False, 'rationale': rationale}


print('Decision model loaded.')
print('Axes:', AXES)
print('Hybrid override fires when: task_complexity >=', HYBRID_COMPLEXITY_MIN,
      'AND (latency_tolerance <=', HYBRID_CONSTRAINT_MAX, 'OR cost_ceiling <=', HYBRID_CONSTRAINT_MAX, ')')

Decision model loaded.
Axes: ['task_complexity', 'latency_tolerance', 'cost_ceiling', 'risk_tolerance', 'update_frequency']
Hybrid override fires when: task_complexity >= 4 AND (latency_tolerance <= 2 OR cost_ceiling <= 2 )


## Walmart Use Case 1: Automated Returns Processing

**Business context:** Walmart processes approximately 1 million returns per day across all channels.
The current system requires a customer service associate to manually look up the purchase history,
check the return policy for the product category, determine eligibility, and process the refund.
The business wants to automate this for self-service kiosks and the Walmart app.

**Key facts:**
- Policy rules are documented and rarely change (quarterly updates)
- Input is structured: item ID, purchase date, reason code, customer ID
- 85% of cases follow one of six standard patterns
- Latency requirement: response within 3 seconds
- Cost sensitivity: millions of transactions per day
- A wrong decision (approving ineligible return) has direct financial impact

In [4]:
use_case_1 = {
    'name': 'Automated Returns Processing',
    'scores': {
        'task_complexity':   2,  # mostly deterministic policy lookup
        'latency_tolerance': 1,  # 3 second hard limit
        'cost_ceiling':      1,  # millions/day -- must be sub-cent
        'risk_tolerance':    1,  # wrong approval = direct financial loss
        'update_frequency':  2,  # policy changes quarterly
    },
    'constraints': {
        'latency_p95_sec': 3,
        'cost_ceiling_per_query_usd': 0.001,
        'daily_volume': 1_000_000,
    }
}

result_1 = recommend_architecture(use_case_1['scores'])
print(f'Use Case 1: {use_case_1["name"]}')
print(f'Total Score: {result_1["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_1["hybrid_override"]}')
print(f'Recommended Architecture: {result_1["architecture"]}')
print(f'Rationale: {result_1["rationale"]}')

Use Case 1: Automated Returns Processing
Total Score: 7 / 25
Hybrid Override Applied: False
Recommended Architecture: Traditional Software
Rationale: Low complexity, strict latency, or low cost ceiling. A rule-based or deterministic pipeline is faster, cheaper, and more predictable.


## Walmart Use Case 2: Supplier Risk Intelligence

**Business context:** Walmart sources from over 100,000 suppliers globally. The procurement team
needs to continuously assess supplier risk across financial stability, geopolitical exposure,
regulatory compliance, and delivery performance. Currently, analysts manually read supplier reports,
news feeds, and financial filings to produce quarterly risk scorecards.

**Key facts:**
- Risk signals come from unstructured sources: news, filings, analyst reports, internal data
- The reasoning required varies significantly by supplier and risk type
- Analysts currently spend 3 hours per supplier per quarter
- Latency tolerance: results needed within 30 minutes of a news event
- Risk assessment logic changes as new risk categories emerge
- A missed risk signal has supply chain consequences, not immediate financial harm

In [5]:
use_case_2 = {
    'name': 'Supplier Risk Intelligence',
    'scores': {
        'task_complexity':   5,  # unstructured multi-source reasoning
        'latency_tolerance': 4,  # 30 min acceptable
        'cost_ceiling':      4,  # per-supplier cost, low volume
        'risk_tolerance':    3,  # missed signal is bad but not immediate
        'update_frequency':  4,  # risk categories evolve frequently
    },
    'constraints': {
        'latency_p95_sec': 1800,
        'cost_ceiling_per_query_usd': 2.00,
        'daily_volume': 500,
    }
}

result_2 = recommend_architecture(use_case_2['scores'])
print(f'Use Case 2: {use_case_2["name"]}')
print(f'Total Score: {result_2["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_2["hybrid_override"]}')
print(f'Recommended Architecture: {result_2["architecture"]}')
print(f'Rationale: {result_2["rationale"]}')

Use Case 2: Supplier Risk Intelligence
Total Score: 20 / 25
Hybrid Override Applied: False
Recommended Architecture: Agent
Rationale: High complexity, dynamic branching, or rapidly changing requirements. An agent is justified -- but implement observability and cost controls from day one.


## Walmart Use Case 3: Store Performance Analytics Reporter

**Business context:** District managers oversee 15-20 Walmart stores each.
Every Monday morning, they manually compile a performance report from five internal systems
(sales data, inventory, staffing, shrinkage, customer satisfaction) and write a summary
with recommended actions for store managers.

**Key facts:**
- All five source systems have structured API access
- The report format is semi-standardised but the narrative varies by week
- District managers spend 2 hours on this every Monday
- Latency tolerance: report must be ready by 8am Monday
- The KPIs tracked are stable but the recommended actions depend on context
- Wrong recommendations are reviewed before action -- human in the loop exists

In [6]:
use_case_3 = {
    'name': 'Store Performance Analytics Reporter',
    'scores': {
        'task_complexity':   3,  # structured data pull + narrative generation
        'latency_tolerance': 5,  # overnight batch acceptable
        'cost_ceiling':      3,  # moderate -- weekly per district manager
        'risk_tolerance':    3,  # human reviews before action
        'update_frequency':  3,  # KPIs stable, narrative context varies
    },
    'constraints': {
        'latency_p95_sec': 3600,
        'cost_ceiling_per_query_usd': 0.50,
        'daily_volume': 50,
    }
}

result_3 = recommend_architecture(use_case_3['scores'])
print(f'Use Case 3: {use_case_3["name"]}')
print(f'Total Score: {result_3["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_3["hybrid_override"]}')
print(f'Recommended Architecture: {result_3["architecture"]}')
print(f'Rationale: {result_3["rationale"]}')

Use Case 3: Store Performance Analytics Reporter
Total Score: 17 / 25
Hybrid Override Applied: False
Recommended Architecture: Workflow (Chain)
Rationale: Moderate complexity with known steps. A fixed-sequence workflow gives you LLM capability without agent overhead.


## Walmart Use Case 4: Customer Service Live Chat Assistant

**Business context:** Walmart handles approximately 2 million customer service interactions per day
across chat, app, and in-store kiosks. The majority of queries are simple and structured:
order status, return eligibility, store hours, and price lookups. However, approximately
15% of queries involve complex, multi-intent scenarios -- disputed charges on partial orders,
damaged-item claims requiring policy interpretation, and price match disputes with promotional
conditions. A pure workflow cannot handle the complex 15%. A pure agent is too expensive and
too slow for the simple 85% at this volume.

**Key facts:**
- Live chat requires a response within 3 seconds to avoid customer drop-off
- At 2M interactions per day, every cent of per-query cost adds USD 20,000 per day
- The complex 15% requires multi-step reasoning: retrieving order history, interpreting
  policy, and generating a personalised resolution -- not a fixed sequence
- Promotions and return policies change weekly, so the routing logic must be current
- A wrong resolution can be corrected by escalation to a human agent -- not zero-risk,
  but not catastrophic

**Why this is not a Workflow:** The complex query segment cannot be handled by a fixed
sequence of steps because the required tools and reasoning path vary per query.

**Why this is not an Agent:** Latency and cost constraints at 2M daily volume make a
full agent architecture economically and operationally infeasible for all queries.

**Why Hybrid:** A query classifier routes each incoming message. Simple queries go to a
deterministic workflow handler. Complex queries are passed to a constrained agent sub-component
with a defined tool set and a strict token budget.

In [7]:
use_case_4 = {
    'name': 'Customer Service Live Chat Assistant',
    'scores': {
        'task_complexity':   4,  # complex reasoning needed for ~15% of queries; NLU required throughout
        'latency_tolerance': 2,  # live chat -- 3-second response window
        'cost_ceiling':      2,  # 2M interactions/day -- per-query cost is operationally critical
        'risk_tolerance':    3,  # wrong answer causes escalation, not direct financial harm
        'update_frequency':  4,  # promotions and return policies change weekly
    },
    'constraints': {
        'latency_p95_sec': 3,
        'cost_ceiling_per_query_usd': 0.002,
        'daily_volume': 2_000_000,
    }
}

result_4 = recommend_architecture(use_case_4['scores'])
print(f'Use Case 4: {use_case_4["name"]}')
print(f'Total Score: {result_4["total_score"]} / 25')
print(f'Hybrid Override Applied: {result_4["hybrid_override"]}')
print(f'Recommended Architecture: {result_4["architecture"]}')
print()
print('Note: total score 15 would place this in Workflow (Chain) by score band alone.')
print('The hybrid override fires because task_complexity=4 and latency_tolerance=2.')
print(f'Rationale: {result_4["rationale"]}')

Use Case 4: Customer Service Live Chat Assistant
Total Score: 15 / 25
Hybrid Override Applied: True
Recommended Architecture: Hybrid (Agent + Workflow)

Note: total score 15 would place this in Workflow (Chain) by score band alone.
The hybrid override fires because task_complexity=4 and latency_tolerance=2.
Rationale: High task complexity coexists with a strict latency or cost constraint. Neither a pure agent nor a pure workflow fits: agent overhead is unacceptable on every step, but deterministic logic cannot handle all branches. A hybrid architecture uses a workflow backbone to route and handle predictable steps, with an agent sub-component invoked only for steps requiring dynamic reasoning.


## Understanding the Hybrid Override

The score band system works well for use cases that sit clearly in one quadrant.
It fails when a use case has internally contradictory requirements -- specifically,
when high task complexity coexists with hard operational constraints.

**Override condition:**
```
task_complexity >= 4  AND  (latency_tolerance <= 2  OR  cost_ceiling <= 2)
```

**What this detects:** A task that genuinely needs dynamic reasoning on some steps
(complexity >= 4) but cannot pay the latency or cost of invoking an agent on every step
(latency or cost <= 2 means the constraint is severe).

**What hybrid means in practice:**

1. A lightweight query classifier (traditional or fast LLM call) routes each incoming request
2. Requests classified as simple go to a deterministic workflow handler -- fast and cheap
3. Requests classified as complex are routed to a constrained agent sub-component with
   a defined tool set, a token budget, and a fallback to human escalation
4. The workflow backbone enforces latency and cost guardrails that the agent sub-component
   cannot exceed

**Why the score band alone is insufficient here:**
Use Case 4 scores 15/25 -- the score band says Workflow (Chain). But a fixed-sequence
workflow cannot handle multi-step reasoning on disputed claims. The score-band result
would lead the team to build something that breaks on 15% of production traffic.

**The override is intentionally narrow.** It fires only when both conditions are present:
complexity is genuinely high AND a hard constraint exists. A complex task without a hard
constraint (Supplier Risk Intelligence: complexity=5, latency=4, cost=4) correctly remains
Agent -- the override does not fire because neither latency nor cost is constrained.

In [8]:
use_cases = [
    (use_case_1, result_1),
    (use_case_2, result_2),
    (use_case_3, result_3),
    (use_case_4, result_4),
]

print(f"{'Use Case':<45} {'Score':>6} {'Override':>8}  {'Recommended Architecture':<30}")
print('-' * 92)
for uc, res in use_cases:
    override_flag = 'YES' if res['hybrid_override'] else 'no'
    print(f'{uc["name"]:<45} {res["total_score"]:>6} {override_flag:>8}  {res["architecture"]:<30}')

print()
print('Key insight: Use Case 4 scores 15/25 -- identical band to Use Case 3.')
print('The hybrid override distinguishes them: Use Case 4 has task_complexity=4')
print('combined with latency_tolerance=2 and cost_ceiling=2.')
print('Score-band alone would incorrectly recommend Workflow for both.')

Use Case                                       Score Override  Recommended Architecture      
--------------------------------------------------------------------------------------------
Automated Returns Processing                       7       no  Traditional Software          
Supplier Risk Intelligence                        20       no  Agent                         
Store Performance Analytics Reporter              17       no  Workflow (Chain)              
Customer Service Live Chat Assistant              15      YES  Hybrid (Agent + Workflow)     

Key insight: Use Case 4 scores 15/25 -- identical band to Use Case 3.
The hybrid override distinguishes them: Use Case 4 has task_complexity=4
combined with latency_tolerance=2 and cost_ceiling=2.
Score-band alone would incorrectly recommend Workflow for both.


## LLM-Assisted Decision Justification

For ARB submission, the recommendation must be defensible in prose, not just a score.
The function below calls GPT-4o-mini to generate a one-paragraph ARB-ready justification
for each use case, grounded in the scored axes.

For hybrid use cases, the justification prompt is extended to require the model to explain
why neither pure alternative is acceptable and how the workflow backbone and agent
sub-component divide responsibility.

In [9]:
def generate_arb_justification(use_case: dict, recommendation: dict) -> str:
    axes_text = '\n'.join([
        f'  {axis}: {score}/5 -- {AXIS_DESCRIPTIONS[axis]}'
        for axis, score in use_case['scores'].items()
    ])

    if recommendation.get('hybrid_override'):
        hybrid_instruction = (
            '5. Explain why a pure workflow is insufficient for the complex query segment.\n'
            '6. Explain why a pure agent is infeasible given the latency and cost constraints.\n'
            '7. Describe how the workflow backbone and agent sub-component divide responsibility.'
        )
        extra_note = (
            f'\n\nNote: This recommendation is the result of a hybrid override. '
            f'The total score of {recommendation["total_score"]} would place this use case in the '
            f'Workflow (Chain) band, but the combination of task_complexity >= 4 and a hard '
            f'operational constraint on latency or cost makes a pure workflow insufficient.'
        )
    else:
        hybrid_instruction = '5. Is written in formal ARB language -- no hedging, no bullet points'
        extra_note = ''

    prompt = (
        f'You are a senior AI architect writing a formal architecture decision note for a Walmart ARB review.\n\n'
        f'Use case: {use_case["name"]}\n'
        f'Recommended architecture: {recommendation["architecture"]}\n'
        f'Total decision score: {recommendation["total_score"]} / 25\n'
        f'{extra_note}\n\n'
        f'Axis scores:\n{axes_text}\n\n'
        f'Write a single concise paragraph (4-6 sentences) that:\n'
        f'1. States the recommended architecture and why\n'
        f'2. Calls out the most influential axis scores\n'
        f'3. Names the primary risk of the alternative architectures\n'
        f'4. Is written in formal ARB language -- no hedging, no bullet points\n'
        f'{hybrid_instruction}'
    )
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3,
        max_tokens=350,
    )
    return response.choices[0].message.content.strip()


print('Generating ARB justifications...')
for uc, res in use_cases:
    justification = generate_arb_justification(uc, res)
    uc['arb_justification'] = justification
    print(f'\n--- {uc["name"]} ---')
    print(justification)

Generating ARB justifications...

--- Automated Returns Processing ---
After careful consideration, the recommended architecture for the Automated Returns Processing use case is Traditional Software, which aligns with the specific requirements and constraints of the project. The most influential axis scores include latency_tolerance and risk_tolerance, both rated at 1/5, indicating that the system must operate with minimal delay and that a high degree of precision is required in the output. The primary risk associated with alternative architectures, such as microservices or serverless solutions, lies in their potential inability to meet the stringent requirements for response time and output accuracy, which could compromise the overall effectiveness of the returns processing system. Given the complexity of the task and the relatively infrequent updates to the business logic, a Traditional Software approach is deemed the most suitable for ensuring reliable and efficient processing of re

# Happy Learning